# 03 - Engineered Features, Character-level Model, Clustering, SOTA

Dataset: **PhiUSIIL Phishing URL Dataset** (UCI id 967).
Target is `label`: 1 = legitimate, 0 = phishing.

Notebook 01 established the leakage-clean feature set and the split
conventions. This notebook builds on those outputs. Here we:

1. reload the data and NB 01's feature manifest (single source of truth)
2. engineer new lexical features NB 01's columns don't already encode
   (entropy, brand-impersonation distance, punycode/shortener flags)
3. train a character-level model on the *raw URL string only* — the
   leakage-immune "hard mode" comparison point
4. cluster the phishing-only rows into attack "families"
5. assemble the SOTA comparison table

The tabular classical models (LR / RF / XGBoost) live in notebook 02.
The overall plan is in `docs/project_plan.md`.

In [5]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Section-specific heavy deps (torch, tldextract, hdbscan, rapidfuzz)
# are imported at the top of their own sections to keep setup light.

DATA_PATH = Path("../data/phiusiil.csv")
MANIFEST_PATH = Path("../data/feature_columns.json")

In [6]:
df = pd.read_csv(DATA_PATH)
manifest = json.loads(MANIFEST_PATH.read_text())

target = manifest["target"]
clean_features = manifest["clean_numeric_features"]
group_col = manifest["split_group_column"]

# Same pre-split cleaning NB 02 uses, so our numbers are comparable.
# We keep URL and Domain as columns (raw material for the char model,
# engineered features and grouping) even though they're not features.
if manifest["drop_duplicate_urls"]:
    before = len(df)
    df = df.drop_duplicates(subset="URL").reset_index(drop=True)
    print(f"dropped {before - len(df)} duplicate-URL rows")

y = df[target]

print(f"rows: {len(df):,}")
print(f"clean feature count: {len(clean_features)}")
print(f"group column for splits: {group_col}")

dropped 425 duplicate-URL rows
rows: 235,370
clean feature count: 49
group column for splits: Domain


## Registered-domain grouping decision

NB 01 split on the exact `Domain` string and flagged (§1) that this
*under*-groups subdomains: `ipfs.io`, `gateway.ipfs.io` and
`cf-ipfs.com` are the same service but count as three domains, so
related rows can still land on both sides of a train/test split. The
stricter alternative groups by **registered domain** (eTLD+1 via
`tldextract`), which collapses those into one group.

This matters because the char-model (§4), the clustering (§5) and
NB 02's tabular models must all use the *same* split, or the SOTA
table compares numbers produced under different conditions. Here we
measure how much stricter grouping changes (a) the group count and
(b) a quick logistic-regression score, then pick one convention and
tell the team.

In [9]:
import tldextract

# eTLD+1, e.g. "gateway.ipfs.io" -> "ipfs.io". Uses a bundled public
# suffix list; suppress the network refresh for reproducibility.
extractor = tldextract.TLDExtract(suffix_list_urls=())


def registered_domain(host: str) -> str:
    ext = extractor(str(host))
    # fall back to the raw host for IP-address URLs (no eTLD+1)
    return ext.top_domain_under_public_suffix or str(host)


df["RegDomain"] = df["Domain"].map(registered_domain)

n_exact = df["Domain"].nunique()
n_reg = df["RegDomain"].nunique()
print(f"exact Domain groups:     {n_exact:,}")
print(f"registered-domain groups: {n_reg:,}")
print(f"collapsed by {n_exact - n_reg:,} groups")
print()
# how many rows share a registered domain with another row?
reg_counts = df["RegDomain"].value_counts()
reg_repeated = reg_counts[reg_counts > 1]
print(
    f"rows sharing a registered domain: "
    f"{reg_repeated.sum():,} ({reg_repeated.sum() / len(df):.1%})"
)
reg_counts.head(10)

exact Domain groups:     220,086
registered-domain groups: 175,509
collapsed by 44,577 groups

rows sharing a registered domain: 65,825 (28.0%)


RegDomain
web.app            5718
firebaseapp.com    5546
repl.co            3746
weeblysite.com     3079
ipfs.io            1552
workers.dev        1423
square.site        1187
dweb.link           959
xsph.ru             928
pantheonsite.io     891
Name: count, dtype: int64

In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler


def quick_logreg(X, y, groups):
    """Basic logistic regression under a grouped split; return acc, F1."""
    gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
    train_idx, test_idx = next(gss.split(X, y, groups=groups))
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)
    model = LogisticRegression(max_iter=1000, class_weight="balanced")
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    return accuracy_score(y_test, preds), f1_score(y_test, preds)


X = df[clean_features]
acc_exact, f1_exact = quick_logreg(X, y, groups=df["Domain"])
acc_reg, f1_reg = quick_logreg(X, y, groups=df["RegDomain"])

print(f"exact-Domain split:      acc={acc_exact:.4f}  f1={f1_exact:.4f}")
print(f"registered-domain split: acc={acc_reg:.4f}  f1={f1_reg:.4f}")

exact-Domain split:      acc=0.9995  f1=0.9996
registered-domain split: acc=0.9990  f1=0.9992


In [10]:
# optional: treat multi-tenant PaaS suffixes as public, so
# victim-a.web.app and victim-b.web.app become separate groups
extractor_strict = tldextract.TLDExtract(
    suffix_list_urls=(), include_psl_private_domains=True
)
df["RegDomainStrict"] = df["Domain"].map(
    lambda h: (e := extractor_strict(str(h))).top_domain_under_public_suffix
    or str(h)
)
print(f"strict (PaaS-aware) groups: {df['RegDomainStrict'].nunique():,}")

strict (PaaS-aware) groups: 197,709


In [11]:
acc_strict, f1_strict = quick_logreg(X, y, groups=df["RegDomainStrict"])
print(f"PaaS-aware split: acc={acc_strict:.4f}  f1={f1_strict:.4f}")

PaaS-aware split: acc=0.9993  f1=0.9994


**Decision.** We compared three train/test grouping strictnesses:

| Grouping | Groups | acc | F1 |
|---|---|---|---|
| exact `Domain` (NB 01's choice) | 220,086 | 0.9995 | 0.9996 |
| PaaS-aware eTLD+1 (PSL-private on) | 197,709 | 0.9993 | 0.9994 |
| naive eTLD+1 | 175,509 | 0.9990 | 0.9992 |

Accuracy is essentially invariant (0.9990–0.9995) across a 44,577-group
range of strictness — more evidence for NB 01's finding that the
dataset is easy regardless of how conservatively the split is drawn.
The drop is monotonic with strictness, matching what leakage would
predict, but negligible in size.

We adopt **PaaS-aware registered-domain grouping** (`tldextract` with
`include_psl_private_domains=True`, `top_domain_under_public_suffix`):
it groups true subdomains together while keeping independent tenants on
shared platforms (`web.app`, `firebaseapp.com`, ...) separate, so it
best approximates "distinct site owner" without the over-grouping that
naive eTLD+1 introduces.

**Convention (pending Jason's sign-off for NB 02):** drop duplicate
URLs, then `GroupShuffleSplit` on `RegDomainStrict`, `random_state=42`.
Because the score is grouping-invariant, if NB 02 is already built on
exact `Domain` we keep that and report the invariance instead of
reworking.

## Character-level model on the raw URL

The tabular models (NB 02) and everything above use page-content and
lexical features that make PhiUSIIL trivially easy (~99.9%). This model
sees **only the raw URL string** — no engineered features, no page
content, not `URLSimilarityIndex`. It cannot leak, because it has
access to nothing but the characters of the URL itself.

That makes it the project's honest "hard mode" number and the direct
comparison point for the URL-only lexical literature (Sahingoz et al.,
2019, reached 97.98% with an NLP-feature Random Forest; §7). We
tokenize each URL at the character level, embed, and pass it through a
small 1-D CNN with a sigmoid output, trained with class weights under
the same PaaS-aware grouped split adopted above.

In [ ]:
# Build a character vocabulary from the training URLs only (fit on
# train to avoid leaking test-set characters into the vocab). Reserve
# 0 for padding and 1 for unknown/out-of-vocab characters.
MAX_LEN = 200  # covers ~99th pct of URL lengths; longer URLs truncated

urls = df["URL"].astype(str).values
labels = df[target].values

lengths = np.array([len(u) for u in urls])
print(f"URL length: median={np.median(lengths):.0f}  "
      f"95th={np.percentile(lengths, 95):.0f}  "
      f"99th={np.percentile(lengths, 99):.0f}  max={lengths.max()}")
print(f"URLs longer than MAX_LEN={MAX_LEN}: "
      f"{(lengths > MAX_LEN).sum()} ({(lengths > MAX_LEN).mean():.2%})")

URL length: median=28  95th=74  99th=145  max=6097
URLs longer than MAX_LEN=200: 1427 (0.61%)
